# 04. Evaluation — 모델 평가

## Google Colab 환경 설정

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "kaggle_secrets" in sys.modules or "/kaggle" in sys.executable or __import__("os").path.exists("/kaggle/input")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    !git clone https://github.com/Lunarian0928/youth_bio_global_portfolio.git
    %cd youth_bio_global_portfolio/oct_retinal_segmentation

    !pip install -q segmentation-models-pytorch albumentations

elif IN_KAGGLE:
    import os

    !git clone https://github.com/Lunarian0928/youth_bio_global_portfolio.git
    os.chdir("/kaggle/working/youth_bio_global_portfolio/oct_retinal_segmentation")
    sys.path.insert(0, "/kaggle/working/youth_bio_global_portfolio/oct_retinal_segmentation")

    !pip install -q segmentation-models-pytorch albumentations


## Imports

In [ ]:
import os
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from src.dataset import OCTRetinalDataset
from src.model import build_model
from src.utils import dice_score, iou_score


## Config

In [ ]:
if IN_COLAB:
    DATA_ROOT = "/content/drive/MyDrive/data/OCT5k"
    TRIALS_CSV = "/content/drive/MyDrive/checkpoints/search/optuna_trials.csv"
    CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints/search"
    SPLITS_DIR = "/content/drive/MyDrive/data/splits"
    RESULTS_DIR = "/content/drive/MyDrive/checkpoints"
elif IN_KAGGLE:
    DATA_ROOT = "/kaggle/input/oct5k-retinal/OCT5k/data/OCT5k"
    TRIALS_CSV = "/kaggle/input/oct5k-retinal/OCT5k/checkpoints/search/optuna_trials.csv"
    CHECKPOINT_DIR = "/kaggle/input/oct5k-retinal/OCT5k/checkpoints/search"
    SPLITS_DIR = "/kaggle/input/oct5k-retinal/OCT5k/data/splits"
    RESULTS_DIR = "/kaggle/working"
else:
    DATA_ROOT = "../data/OCT5k"
    TRIALS_CSV = "../checkpoints/search/optuna_trials.csv"
    CHECKPOINT_DIR = "../checkpoints/search"
    SPLITS_DIR = "../data/splits"
    RESULTS_DIR = "../checkpoints"

GRADING = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 학습된 모델 로드

In [ ]:
import pandas as pd

# optuna_trials.csv에서 best trial 찾기
df_trials = pd.read_csv(TRIALS_CSV)
best_trial = df_trials.loc[df_trials["value"].idxmax()]
best_trial_number = int(best_trial["number"])
print(f"Best trial: {best_trial_number}, val_dice: {best_trial['value']:.4f}")
print(f"Best params: lr={best_trial['params_lr']:.6f}, batch_size={int(best_trial['params_batch_size'])}, weight_decay={best_trial['params_weight_decay']:.6f}")

# best trial의 .pt 파일 로드
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/trial{best_trial_number}_best.pt"
print(f"체크포인트 로드: {CHECKPOINT_PATH}")

model = build_model()
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print("모델 로드 완료")


## 테스트 데이터 로드

In [ ]:
TEST_CSV = os.path.join(SPLITS_DIR, "test.csv")
test_dataset = OCTRetinalDataset.from_csv(TEST_CSV)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)
print("테스트 샘플 수:", len(test_dataset))


## Inference

In [ ]:
all_preds = []
all_masks = []
all_images = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.append(preds.cpu())
        all_masks.append(masks.cpu())
        all_images.append(images.cpu())

all_preds = torch.cat(all_preds, dim=0)
all_masks = torch.cat(all_masks, dim=0)
all_images = torch.cat(all_images, dim=0)
print("추론 완료. 예측 샘플 수:", len(all_preds))


## 평가지표 계산 (Dice, IoU, Sensitivity, Specificity)

In [ ]:
NUM_CLASSES = 6
CLASS_NAMES = ["배경", "ILM", "OPL", "IS/OS", "IBRPE", "OBRPE"]

results = []
for c in range(NUM_CLASSES):
    pred_c = all_preds == c
    mask_c = all_masks == c

    tp = (pred_c & mask_c).sum().item()
    fp = (pred_c & ~mask_c).sum().item()
    fn = (~pred_c & mask_c).sum().item()
    tn = (~pred_c & ~mask_c).sum().item()

    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    iou = tp / (tp + fp + fn + 1e-8)
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)

    results.append(
        {
            "클래스": CLASS_NAMES[c],
            "Dice Score": round(dice, 4),
            "IoU": round(iou, 4),
            "Sensitivity": round(sensitivity, 4),
            "Specificity": round(specificity, 4),
        }
    )

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

RESULTS_PATH = os.path.join(RESULTS_DIR, "evaluation_results.csv")
df_results.to_csv(RESULTS_PATH, index=False)
print(f"평가 결과 저장 완료: {RESULTS_PATH}")


## 예측 결과 시각화

In [ ]:
NUM_SAMPLES = 6
fig, axes = plt.subplots(NUM_SAMPLES, 3, figsize=(12, NUM_SAMPLES * 4))
col_titles = ["OCT 이미지", "Ground Truth", "예측 마스크"]

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=14, fontweight="bold")

for i in range(NUM_SAMPLES):
    image = all_images[i].squeeze().numpy()
    mask = all_masks[i].numpy()
    pred = all_preds[i].numpy()

    axes[i, 0].imshow(image, cmap="gray")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask, cmap="tab10", vmin=0, vmax=5)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(pred, cmap="tab10", vmin=0, vmax=5)
    axes[i, 2].axis("off")

plt.tight_layout()
VIZ_PATH = os.path.join(RESULTS_DIR, "evaluation_result.png")
plt.savefig(VIZ_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"시각화 저장 완료: {VIZ_PATH}")
